# Extracción Sentinel-2 de las cuatro parcelas

Procesamiento automático por lotes, revisión de calidad, índices, cobertura, series semanales y comparativas.

## Cómo utilizarlo
Ejecutar de arriba abajo en Colab con acceso a Drive y Earth Engine. Mantener primero los parámetros de referencia. Esta copia no incluye salidas: al ejecutarla se consulta Earth Engine y se escriben los resultados en Drive. Las explicaciones de las celdas describen el procedimiento, no resultados.

## Dependencia
Completar antes el notebook 02 y disponer de catalogo_parcelas.csv y los cuatro GeoJSON. El identificador del proyecto de Earth Engine no se guarda en el notebook: se lee del secreto `PROYECTO_EE` de Colab o se escribe al ejecutar (ver apartado 8).

## Alcance
El intervalo es provisional. No se confirma patata en esas fechas ni se generan dosis de riego. No se incorporan históricos. Se conservan observaciones rechazadas y semanas vacías para permitir revisar la selección.


## 1. Dependencia de Earth Engine

Instala la API Python en Colab. No descarga todavía imágenes ni modifica los datos de las parcelas.


In [ ]:
%pip -q install earthengine-api

## 2. Drive, librerías y rutas de entrada

Monta Drive y localiza catalogo_parcelas.csv. Si no existe, hay que completar el notebook 02. Importa también datetime y timezone para registrar la ejecución en UTC.


In [ ]:
import json
import unicodedata
from pathlib import Path

import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

base_path = Path(
    "/content/drive/MyDrive/TFG_PATATAS_TRABAJO"
)

parcelas_path = base_path / "parcelas"
catalogo_path = parcelas_path / "catalogo_parcelas.csv"

if not catalogo_path.exists():
    raise FileNotFoundError(
        "No se encuentra catalogo_parcelas.csv. "
        "Ejecuta primero el notebook de preparación."
    )
from datetime import datetime, timezone


## 3. Lectura y comprobación del catálogo

Comprueba las columnas esenciales, datos ausentes e identificadores repetidos. El catálogo contiene cuatro parcelas en la versión actual. Esta revisión no confirma el cultivo de la campaña provisional.


In [ ]:
catalogo = pd.read_csv(
    catalogo_path,
    dtype="string"
)

columnas_necesarias = {
    "parcela_id",
    "nombre",
    "municipio",
    "variedad",
    "archivo_geometria"
}

faltantes = columnas_necesarias - set(catalogo.columns)

if faltantes:
    raise ValueError(
        f"Faltan columnas en el catálogo: {sorted(faltantes)}"
    )

if catalogo[list(columnas_necesarias)].isna().any().any():
    raise ValueError(
        "Hay datos vacíos en las columnas de identificación o geometría."
    )

if not catalogo["parcela_id"].is_unique:
    raise ValueError(
        "El catálogo contiene identificadores repetidos."
    )

display(
    catalogo[
        [
            "parcela_id",
            "nombre",
            "municipio",
            "variedad",
            "archivo_geometria"
        ]
    ]
)

## 4. Carga de una parcela

Define la normalización de nombres y la lectura del GeoJSON asociado. Revisa el tipo Polygon y el cierre de sus anillos. La validación topológica completa corresponde al notebook de preparación.


In [ ]:
def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = (
        texto.strip()
        .casefold()
        .replace("_", " ")
        .replace("-", " ")
    )

    return " ".join(texto.split())


def cargar_parcela(nombre_o_id):
    busqueda = normalizar_nombre(nombre_o_id)

    mascara = (
        catalogo["nombre"].map(normalizar_nombre).eq(busqueda)
        | catalogo["parcela_id"].map(normalizar_nombre).eq(busqueda)
    )

    coincidencias = catalogo.loc[mascara]

    if len(coincidencias) != 1:
        disponibles = ", ".join(catalogo["nombre"].tolist())

        raise ValueError(
            f"Parcela no reconocida: {nombre_o_id}. "
            f"Disponibles: {disponibles}"
        )

    parcela = coincidencias.iloc[0].to_dict()
    archivo = parcelas_path / parcela["archivo_geometria"]

    if not archivo.exists():
        raise FileNotFoundError(
            f"No se encuentra {archivo.name}."
        )

    geometria = json.loads(
        archivo.read_text(encoding="utf-8")
    )

    if geometria.get("type") != "Polygon":
        raise ValueError(
            "El archivo debe contener una geometría Polygon."
        )

    anillos = geometria.get("coordinates", [])

    if not anillos:
        raise ValueError("La geometría no contiene coordenadas.")

    for anillo in anillos:
        if len(anillo) < 4 or anillo[0] != anillo[-1]:
            raise ValueError(
                "La geometría contiene un anillo incompleto o no cerrado."
            )

    parcela["geometria"] = geometria

    return parcela

## 5. Validación de fechas reales

Exige texto en formato YYYY-MM-DD, comprueba la existencia de la fecha mediante datetime.date y exige inicio anterior al fin. Devuelve objetos date, que se convierten con isoformat() al consultar y guardar.


In [ ]:
import re
from datetime import date


def validar_fecha(valor, nombre_campo):
    if not isinstance(valor, str):
        raise TypeError(
            f"{nombre_campo} debe introducirse como texto "
            'entre comillas, por ejemplo: "2026-04-01".'
        )

    if re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", valor) is None:
        raise ValueError(
            f"{nombre_campo}: '{valor}' no tiene el formato YYYY-MM-DD."
        )

    try:
        fecha = date.fromisoformat(valor)
    except ValueError as error:
        raise ValueError(
            f"{nombre_campo}: '{valor}' no es una fecha real del calendario."
        ) from error

    return fecha


def validar_intervalo(fecha_inicio, fecha_fin):
    inicio = validar_fecha(fecha_inicio, "FECHA_INICIO")
    fin = validar_fecha(fecha_fin, "FECHA_FIN")

    if inicio >= fin:
        raise ValueError(
            "FECHA_INICIO debe ser anterior a FECHA_FIN."
        )

    return inicio, fin




## 6. Pruebas de fechas e intervalos

Prueba fechas válidas, años bisiestos, fechas imposibles, formatos incorrectos y rangos invertidos. No cambia el intervalo de la extracción. Si una prueba falla, resolverla antes de consultar Earth Engine.


In [ ]:
casos_fecha = [
    ("2026-04-01", True),
    ("2024-02-29", True),
    ("2026-02-29", False),
    ("2026-02-30", False),
    ("2026-04-31", False),
    ("2026-13-01", False),
    ("2026-00-10", False),
    ("2026-04-00", False),
    ("0000-04-01", False),
    ("2026-4-1", False),
    ("01/04/2026", False),
    ("2026-04-01T10:00:00", False),
    ("", False),
    (None, False)
]

for valor, deberia_ser_valida in casos_fecha:
    try:
        validar_fecha(valor, "FECHA_PRUEBA")
        es_valida = True
    except (TypeError, ValueError):
        es_valida = False

    assert es_valida == deberia_ser_valida, (
        f"Resultado inesperado para {valor!r}"
    )

print("Pruebas de fechas superadas.")


casos_intervalo = [
    ("2026-04-01", "2026-10-01", True),
    ("2026-04-01", "2026-04-01", False),
    ("2026-10-01", "2026-04-01", False),
    ("2026-02-30", "2026-10-01", False)
]

for inicio_prueba, fin_prueba, deberia_ser_valido in casos_intervalo:
    try:
        validar_intervalo(inicio_prueba, fin_prueba)
        es_valido = True
    except (TypeError, ValueError):
        es_valido = False

    assert es_valido == deberia_ser_valido, (
        f"Resultado inesperado para "
        f"{inicio_prueba!r} → {fin_prueba!r}"
    )

print("Pruebas de intervalos superadas.")

## 7. Configuración central del lote

Define fechas y parámetros comunes. El inicio 2026-04-01 se incluye y el fin 2026-10-01 se excluye. Es un periodo provisional, no una campaña de patata confirmada. El umbral de nube es 20 y se conservan probabilidades inferiores; la cobertura mínima provisional es 70 %. La escala de evaluación es 20 m. Mantener parámetros iguales para contrastar con las primeras pruebas; cualquier cambio exige revisar los resultados. Cambiar la escala no crea información de mayor resolución.


In [ ]:
FECHA_INICIO = "2026-04-01"
FECHA_FIN = "2026-10-01"
PARAMETROS = {
    "umbral_nube": 20,
    "cobertura_minima_pct": 70.0,
    "escala_m": 20,
    "indices": ["NDVI", "NDMI", "MSI", "EVI", "SAVI"],
    "periodo_provisional": True,
    "cultivo_confirmado": False
}
inicio, fin = validar_intervalo(FECHA_INICIO, FECHA_FIN)
if not 0 <= PARAMETROS["umbral_nube"] <= 100:
    raise ValueError("Umbral de nube fuera de rango.")
if not 0 <= PARAMETROS["cobertura_minima_pct"] <= 100:
    raise ValueError("Cobertura mínima fuera de rango.")
if PARAMETROS["escala_m"] <= 0:
    raise ValueError("La escala debe ser positiva.")
print("Intervalo validado:", inicio, fin)


## 8. Autenticación e inicialización

Autoriza Earth Engine con tu cuenta e inicializa tu proyecto de Earth Engine. El identificador del proyecto se lee del secreto `PROYECTO_EE` de Colab (icono de la llave, con acceso a este notebook concedido); si no existe, se pide al ejecutar sin mostrarlo en pantalla. `force=False` reutiliza la autenticación existente si la hay. La conexión se realiza una vez antes del lote, no una vez por parcela. No compartir códigos ni tokens.


In [ ]:
import ee
from getpass import getpass

# El identificador del proyecto de Earth Engine no se guarda en el notebook.
# 1) Se intenta leer del secreto PROYECTO_EE de Colab (icono de la llave, con acceso a este notebook).
# 2) Si no está disponible, se pide al ejecutar sin mostrarlo en pantalla.
PROYECTO_EE = None
try:
    from google.colab import userdata
    PROYECTO_EE = (userdata.get("PROYECTO_EE") or "").strip()
except Exception as error:
    print("No se pudo leer el secreto PROYECTO_EE:", type(error).__name__, "-", error)

if not PROYECTO_EE:
    PROYECTO_EE = getpass("Identificador del proyecto de Earth Engine: ").strip()

if not PROYECTO_EE:
    raise ValueError("Falta el identificador del proyecto de Earth Engine.")

ee.Authenticate(
    auth_mode="colab",
    force=False
)

ee.Initialize(
    project=PROYECTO_EE
)

print("Earth Engine inicializado correctamente.")


## 9. Preparación del conjunto de parcelas

Carga todas las entradas del catálogo y muestra el orden de procesamiento. No hay que cambiar NOMBRE_PARCELA. Si el catálogo está vacío, se detiene antes de extraer datos.


In [ ]:
parcelas_lote = [cargar_parcela(pid) for pid in catalogo["parcela_id"]]
if not parcelas_lote:
    raise ValueError("El catálogo está vacío.")
print("Parcelas del lote:", [p["nombre"] for p in parcelas_lote])


## 10. Agregación semanal sin interpolación

crear_resumen_semanal agrupa de lunes incluido a lunes siguiente excluido. Calcula la media aritmética de las medias de las imágenes seleccionadas, no una media continua de todos los días ni una media espacial combinada de todos los píxeles. Incluye semanas sin observaciones, con índices ausentes; no rellena con ceros ni interpola. Conserva fechas e identificadores, y marca semanas parciales en los extremos. Para uso diario futuro, no se debe adelantar al lunes información adquirida más tarde esa semana.


In [ ]:
def crear_resumen_semanal(
    datos,
    indices,
    fecha_inicio,
    fecha_fin
):
    datos = datos.copy()

    datos["date"] = pd.to_datetime(
        datos["date"],
        errors="raise"
    )

    inicio_periodo = pd.Timestamp(fecha_inicio)
    fin_periodo = pd.Timestamp(fecha_fin)

    if inicio_periodo >= fin_periodo:
        raise ValueError(
            "El inicio debe ser anterior al fin."
        )

    if datos.empty:
        raise ValueError(
            "No hay observaciones para preparar el resumen."
        )

    if datos["parcela_id"].nunique() != 1:
        raise ValueError(
            "La función debe recibir una sola parcela."
        )

    if not (
        datos["date"].ge(inicio_periodo)
        & datos["date"].lt(fin_periodo)
    ).all():
        raise ValueError(
            "Hay observaciones fuera del intervalo."
        )

    seleccion = datos[
        "seleccion_tecnica_provisional"
    ]

    if seleccion.isna().any() or not seleccion.isin(
        [True, False]
    ).all():
        raise ValueError(
            "La selección técnica debe contener booleanos."
        )

    primer_lunes = (
        inicio_periodo
        - pd.Timedelta(days=inicio_periodo.weekday())
    )

    ultimo_dia = fin_periodo - pd.Timedelta(days=1)

    ultimo_lunes = (
        ultimo_dia
        - pd.Timedelta(days=ultimo_dia.weekday())
    )

    semanas = pd.date_range(
        primer_lunes,
        ultimo_lunes,
        freq="W-MON",
        name="semana_inicio"
    )

    todos = (
        datos
        .sort_values(["date", "image_id"])
        .set_index("date")
    )

    seleccionados = todos.loc[
        todos["seleccion_tecnica_provisional"].eq(True)
    ]

    argumentos = {
        "rule": "W-MON",
        "closed": "left",
        "label": "left"
    }

    semanal = pd.DataFrame(index=semanas)

    semanal["n_imagenes_con_datos"] = (
        todos.resample(**argumentos)
        .size()
        .reindex(semanas, fill_value=0)
        .astype(int)
    )

    semanal["n_imagenes_seleccionadas"] = (
        seleccionados.resample(**argumentos)
        .size()
        .reindex(semanas, fill_value=0)
        .astype(int)
    )

    semanal["n_imagenes_no_seleccionadas"] = (
        semanal["n_imagenes_con_datos"]
        - semanal["n_imagenes_seleccionadas"]
    )

    columnas_medias = [
        f"{indice}_mean"
        for indice in indices
    ]

    medias = (
        seleccionados[columnas_medias]
        .resample(**argumentos)
        .mean()
        .reindex(semanas)
    )

    medias = medias.rename(
        columns={
            f"{indice}_mean": f"{indice}_media_semanal"
            for indice in indices
        }
    )

    semanal = semanal.join(medias)

    semanal["cobertura_minima_utilizada_pct"] = (
        seleccionados["cobertura_minima_indices_pct"]
        .resample(**argumentos)
        .min()
        .reindex(semanas)
    )

    fechas_texto = pd.Series(
        seleccionados.index.strftime("%Y-%m-%d"),
        index=seleccionados.index,
        name="fecha_observacion"
    )

    semanal["fechas_utilizadas"] = (
        fechas_texto
        .resample(**argumentos)
        .agg(lambda valores: "|".join(valores.tolist()))
        .reindex(semanas)
        .fillna("")
    )

    semanal["imagenes_utilizadas"] = (
        seleccionados["image_id"]
        .resample(**argumentos)
        .agg(lambda valores: "|".join(valores.astype(str)))
        .reindex(semanas)
        .fillna("")
    )

    semanal["semana_fin_excluida"] = (
        semanal.index + pd.Timedelta(days=7)
    )

    semanal["semana_parcial_en_periodo"] = (
        (semanal.index < inicio_periodo)
        | (semanal["semana_fin_excluida"] > fin_periodo)
    )

    semanal["tiene_observaciones_seleccionadas"] = (
        semanal["n_imagenes_seleccionadas"] > 0
    )

    semanal["origen_dato"] = (
        semanal["tiene_observaciones_seleccionadas"]
        .map({
            True: "agregacion_observaciones_satelitales",
            False: "sin_observaciones_seleccionadas"
        })
    )

    semanal["interpolado"] = False
    semanal["metodo_agregacion"] = "media_aritmetica"

    semanal["parcela_id"] = datos["parcela_id"].iloc[0]
    semanal["periodo_provisional"] = (
        datos["periodo_provisional"].iloc[0]
    )

    return semanal.reset_index()

## 11. Función de procesamiento individual

procesar_parcela encapsula geometría, colecciones, tablas y rutas de una parcela. Las funciones internas utilizan ese contexto local; los diccionarios de reduceRegion se llaman parametros_reduccion para no ocultar la configuración general.

### Secuencia interna
1. Validar el intervalo, crear la carpeta y guardar configuración y momento de ejecución.
2. Filtrar Sentinel-2 SR por intersección y fechas; asociar probabilidad de nube por system:index.
3. Excluir píxeles por probabilidad, clases SCL y máscaras de bordes. Estas máscaras no garantizan una eliminación perfecta de nubes y sombras.
4. Guardar el recuento B8 antes y después. Una imagen puede quedar sin píxeles; esos descartes permanecen documentados.
5. Calcular NDVI, NDMI, MSI, EVI y SAVI sobre bandas convertidas a reflectancia con 0.0001; enmascarar denominadores no válidos.
6. Obtener media, desviación, mínimo, máximo, mediana, percentiles y recuento por índice.
7. Calcular superficie válida por índice con pixelArea respecto a una referencia de la parcela en la cuadrícula de evaluación. No confundir cobertura de parcela con porcentaje de píxeles conservados respecto a los disponibles en una imagen.
8. Marcar selección con cobertura mínima suficiente y medias completas, sin borrar las filas no seleccionadas.
9. Detenerse si hay varias imágenes seleccionadas para una fecha, antes del resumen semanal. No resuelve automáticamente ese caso.
10. Crear, guardar y volver a leer el resumen semanal; devolver tablas, resumen y carpeta.

### Archivos y límites
Se escribe en resultados/sentinel2_lotes, separado de la referencia manual. Repetir parcela y periodo actualiza sus archivos. Una ejecución que falle puede dejar archivos individuales parciales. El código actual detiene el lote si no encuentra imágenes emparejadas o imágenes con píxeles válidos. Las estadísticas y el filtro no constituyen un diagnóstico de riego. Los índices de vegetación no son mediciones directas de humedad del suelo.

La evaluación usa escala y proyección indicadas en cada reducción; no interpreta los recuentos de índices y B8 como necesariamente intercambiables. Se conserva el comportamiento del código original, sin cambiar fórmulas ni remuestreo.


In [ ]:
def procesar_parcela(parcela, fecha_inicio, fecha_fin, parametros, base_path):
    parcela_actual = dict(parcela)
    inicio, fin = validar_intervalo(fecha_inicio, fecha_fin)

    PERIODO_PROVISIONAL = parametros["periodo_provisional"]
    CULTIVO_CONFIRMADO_EN_PERIODO = parametros["cultivo_confirmado"]
    INDICES = list(parametros["indices"])

    print("\nPROCESANDO:", parcela_actual["nombre"])

    tipo_periodo = (
        "provisional"
        if PERIODO_PROVISIONAL
        else "confirmado"
    )

    extraccion_path = (
        base_path
        / "resultados"
        / "sentinel2_lotes"
        / tipo_periodo
        / parcela_actual["parcela_id"]
        / f"{inicio.isoformat()}_{fin.isoformat()}"
    )

    extraccion_path.mkdir(
        parents=True,
        exist_ok=True
    )

    print("Carpeta de resultados:")
    print(extraccion_path)

    configuracion = {
        "parcela_id": parcela_actual["parcela_id"],
        "nombre_parcela": parcela_actual["nombre"],
        "municipio": parcela_actual["municipio"],
        "variedad_catalogo": parcela_actual["variedad"],
        "fecha_inicio_incluida": inicio.isoformat(),
        "fecha_fin_excluida": fin.isoformat(),
        "periodo_provisional": PERIODO_PROVISIONAL,
        "cultivo_confirmado_en_periodo": CULTIVO_CONFIRMADO_EN_PERIODO,
        "objetivo": "Prueba técnica de extracción Sentinel-2",
        "parametros": dict(parametros),
        "ejecutado_utc": datetime.now(timezone.utc).isoformat()
    }

    configuracion_path = (
        extraccion_path / "configuracion_extraccion.json"
    )

    configuracion_path.write_text(
        json.dumps(
            configuracion,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    print("Configuración guardada.")

    assert (
        configuracion["parcela_id"]
        == parcela_actual["parcela_id"]
    ), (
        "La configuración pertenece a otra parcela. "
        "Vuelve a ejecutar las celdas de rutas y configuración."
    )

    assert extraccion_path.parent.name == parcela_actual["parcela_id"], (
        "La carpeta de resultados pertenece a otra parcela."
    )

    assert configuracion["fecha_inicio_incluida"] == inicio.isoformat()
    assert configuracion["fecha_fin_excluida"] == fin.isoformat()

    print("Parcela de esta ejecución:", parcela_actual["nombre"])
    print("Identificador:", parcela_actual["parcela_id"])
    print("Carpeta:", extraccion_path)

    polygon = ee.Geometry(
        parcela_actual["geometria"]
    )

    coleccion_prueba = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(polygon)
        .filterDate(
            inicio.isoformat(),
            fin.isoformat()
        )
    )

    numero_imagenes = coleccion_prueba.size().getInfo()

    print("Parcela:", parcela_actual["nombre"])
    print("Inicio incluido:", inicio.isoformat())
    print("Fin excluido:", fin.isoformat())
    print("Imágenes que intersectan la parcela:", numero_imagenes)

    if numero_imagenes == 0:
        print(
            "No hay imágenes para esta consulta. "
            "Revisaremos el intervalo y la geometría."
        )
    else:
        primera_imagen = ee.Image(
            coleccion_prueba
            .sort("system:time_start")
            .first()
        )

        primera_fecha = (
            primera_imagen.date()
            .format("YYYY-MM-dd")
            .getInfo()
        )

        print("Primera fecha disponible:", primera_fecha)

    UMBRAL_PROBABILIDAD_NUBE = parametros["umbral_nube"]

    print(
        "Se conservarán píxeles con probabilidad de nube menor que:",
        UMBRAL_PROBABILIDAD_NUBE
    )

    coleccion_probabilidad_nube = (
        ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
        .filterBounds(polygon)
        .filterDate(
            inicio.isoformat(),
            fin.isoformat()
        )
    )

    condicion_union = ee.Filter.equals(
        leftField="system:index",
        rightField="system:index"
    )

    coleccion_unida = ee.ImageCollection(
        ee.Join.saveFirst(
            matchKey="cloud_probability",
            outer=True
        ).apply(
            primary=coleccion_prueba,
            secondary=coleccion_probabilidad_nube,
            condition=condicion_union
        )
    )

    coleccion_con_nubes = (
        coleccion_unida
        .filter(
            ee.Filter.notNull(["cloud_probability"])
        )
    )

    numero_originales = coleccion_prueba.size().getInfo()
    numero_emparejadas = coleccion_con_nubes.size().getInfo()

    print("Parcela:", parcela_actual["nombre"])
    print("Imágenes originales:", numero_originales)
    print("Imágenes con información de nubes:", numero_emparejadas)
    print(
        "Imágenes sin información de nubes:",
        numero_originales - numero_emparejadas
    )

    if numero_emparejadas == 0:
        raise ValueError(
            "No se han encontrado imágenes con información de nubes. "
            "Revisaremos la disponibilidad antes de continuar."
        )

    def aplicar_mascara_calidad(image):
        image = ee.Image(image)

        probabilidad = (
            ee.Image(image.get("cloud_probability"))
            .select("probability")
        )

        scl = image.select("SCL")

        mascara_probabilidad = (
            probabilidad.lt(UMBRAL_PROBABILIDAD_NUBE)
        )

        mascara_scl = (
            scl.neq(0)
            .And(scl.neq(1))
            .And(scl.neq(3))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
            .And(scl.neq(11))
        )

        mascara_bordes = (
            image.select("B8A").mask()
            .And(image.select("B9").mask())
        )

        return (
            image
            .updateMask(mascara_probabilidad)
            .updateMask(mascara_scl)
            .updateMask(mascara_bordes)
        )

    coleccion_limpia = (
        coleccion_con_nubes
        .map(aplicar_mascara_calidad)
    )

    print("Colección con máscara de calidad preparada.")

    def resumir_calidad_imagen(elemento):
        image = ee.Image(elemento)

        image_limpia = aplicar_mascara_calidad(
            image
        )

        proyeccion = (
            image.select("B8").projection()
        )

        parametros_reduccion = {
            "reducer": ee.Reducer.count(),
            "geometry": polygon,
            "crs": proyeccion,
            "scale": parametros["escala_m"],
            "maxPixels": 1_000_000
        }

        pixeles_antes = (
            image
            .select("B8")
            .reduceRegion(**parametros_reduccion)
            .get("B8")
        )

        pixeles_despues = (
            image_limpia
            .select("B8")
            .reduceRegion(**parametros_reduccion)
            .get("B8")
        )

        return ee.Feature(
            None,
            {
                "date": image.date().format("YYYY-MM-dd"),
                "image_id": image.get("system:index"),
                "pixeles_antes": pixeles_antes,
                "pixeles_despues": pixeles_despues
            }
        )

    lista_imagenes = coleccion_con_nubes.toList(
        coleccion_con_nubes.size()
    )

    resumen_calidad = ee.FeatureCollection(
        lista_imagenes.map(resumir_calidad_imagen)
    )

    datos_calidad = resumen_calidad.getInfo()

    calidad_df = pd.DataFrame([
        feature["properties"]
        for feature in datos_calidad["features"]
    ])

    if calidad_df.empty:
        raise ValueError(
            "La colección emparejada no contiene imágenes para revisar."
        )

    calidad_df["date"] = pd.to_datetime(
        calidad_df["date"]
    )

    for columna in ["pixeles_antes", "pixeles_despues"]:
        calidad_df[columna] = (
            pd.to_numeric(
                calidad_df[columna],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )

    calidad_df = (
        calidad_df
        .sort_values(["date", "image_id"])
        .reset_index(drop=True)
    )

    calidad_df["porcentaje_conservado"] = (
        100
        * calidad_df["pixeles_despues"]
        / calidad_df["pixeles_antes"].replace(0, float("nan"))
    )

    assert (
        calidad_df["pixeles_despues"]
        <= calidad_df["pixeles_antes"]
    ).all(), "La máscara ha aumentado inesperadamente el recuento."

    display(calidad_df)

    print("Imágenes revisadas:", len(calidad_df))
    print(
        "Imágenes con algún píxel válido:",
        int((calidad_df["pixeles_despues"] > 0).sum())
    )
    print(
        "Imágenes sin píxeles válidos:",
        int((calidad_df["pixeles_despues"] == 0).sum())
    )

    calidad_df.to_csv(
        extraccion_path / "revision_calidad_imagenes.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Resumen de calidad guardado.")

    ids_con_datos = (
        calidad_df.loc[
            calidad_df["pixeles_despues"] > 0,
            "image_id"
        ]
        .tolist()
    )

    if not ids_con_datos:
        raise ValueError(
            "No hay imágenes con píxeles válidos para calcular índices."
        )

    coleccion_con_datos = (
        coleccion_limpia
        .filter(
            ee.Filter.inList(
                "system:index",
                ids_con_datos
            )
        )
    )

    numero_con_datos = (
        coleccion_con_datos.size().getInfo()
    )

    print("Imágenes seleccionadas:", numero_con_datos)

    assert numero_con_datos == len(ids_con_datos), (
        "El número de imágenes no coincide con el resumen de calidad."
    )

    def agregar_indices(image):
        image = ee.Image(image)

        reflectancia = (
            image
            .select(["B2", "B4", "B8", "B11"])
            .multiply(0.0001)
        )

        azul = reflectancia.select("B2")
        rojo = reflectancia.select("B4")
        nir = reflectancia.select("B8")
        swir = reflectancia.select("B11")

        denominador_ndvi = nir.add(rojo)

        ndvi = (
            nir.subtract(rojo)
            .divide(denominador_ndvi)
            .updateMask(denominador_ndvi.gt(0))
            .rename("NDVI")
        )

        denominador_ndmi = nir.add(swir)

        ndmi = (
            nir.subtract(swir)
            .divide(denominador_ndmi)
            .updateMask(denominador_ndmi.gt(0))
            .rename("NDMI")
        )

        msi = (
            swir.divide(nir)
            .updateMask(nir.gt(0))
            .rename("MSI")
        )

        denominador_evi = (
            nir
            .add(rojo.multiply(6))
            .subtract(azul.multiply(7.5))
            .add(1)
        )

        evi = (
            nir.subtract(rojo)
            .multiply(2.5)
            .divide(denominador_evi)
            .updateMask(denominador_evi.abs().gt(1e-6))
            .rename("EVI")
        )

        denominador_savi = (
            nir.add(rojo).add(0.5)
        )

        savi = (
            nir.subtract(rojo)
            .multiply(1.5)
            .divide(denominador_savi)
            .updateMask(denominador_savi.gt(0))
            .rename("SAVI")
        )

        return image.addBands([
            ndvi,
            ndmi,
            msi,
            evi,
            savi
        ])

    coleccion_indices = (
        coleccion_con_datos
        .map(agregar_indices)
    )

    print("Colección con índices preparada.")

    ESCALA_ESTADISTICAS_M = parametros["escala_m"]

    reductor_estadisticas = (
        ee.Reducer.mean()
        .combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.median(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.percentile([25, 75]),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.count(),
            sharedInputs=True
        )
    )

    def extraer_estadisticas(elemento):
        image = ee.Image(elemento)

        estadisticas = (
            image.select(INDICES)
            .reduceRegion(
                reducer=reductor_estadisticas,
                geometry=polygon,
                crs=image.select("B8").projection(),
                scale=ESCALA_ESTADISTICAS_M,
                maxPixels=1_000_000
            )
        )

        propiedades = ee.Dictionary({
            "parcela_id": parcela_actual["parcela_id"],
            "nombre_parcela": parcela_actual["nombre"],
            "municipio": parcela_actual["municipio"],
            "variedad_catalogo": parcela_actual["variedad"],
            "date": image.date().format("YYYY-MM-dd"),
            "image_id": image.get("system:index"),
            "cloud_percentage": image.get("CLOUDY_PIXEL_PERCENTAGE"),
            "periodo_provisional": PERIODO_PROVISIONAL,
            "cultivo_confirmado_en_periodo": CULTIVO_CONFIRMADO_EN_PERIODO,
            "fecha_inicio_incluida": inicio.isoformat(),
            "fecha_fin_excluida": fin.isoformat(),
            "escala_estadisticas_m": ESCALA_ESTADISTICAS_M,
            "umbral_probabilidad_nube": UMBRAL_PROBABILIDAD_NUBE
        }).combine(estadisticas)

        return ee.Feature(None, propiedades)

    lista_indices = coleccion_indices.toList(
        coleccion_indices.size()
    )

    resultados_indices = ee.FeatureCollection(
        lista_indices.map(extraer_estadisticas)
    )

    datos_indices = resultados_indices.getInfo()

    filas_indices = [
        feature["properties"]
        for feature in datos_indices["features"]
    ]

    df_indices = pd.DataFrame(filas_indices)

    if df_indices.empty:
        raise ValueError(
            "La extracción no ha generado ninguna fila."
        )

    df_indices["date"] = pd.to_datetime(
        df_indices["date"],
        format="%Y-%m-%d",
        errors="raise"
    )

    df_indices = (
        df_indices
        .sort_values(["date", "image_id"])
        .reset_index(drop=True)
    )

    print("Filas extraídas:", len(df_indices))
    print("Fechas distintas:", df_indices["date"].nunique())
    print("Primera fecha:", df_indices["date"].min().date())
    print("Última fecha:", df_indices["date"].max().date())

    columnas_identificacion = [
        "parcela_id",
        "nombre_parcela",
        "municipio",
        "variedad_catalogo",
        "date",
        "image_id",
        "cloud_percentage",
        "periodo_provisional",
        "cultivo_confirmado_en_periodo",
        "fecha_inicio_incluida",
        "fecha_fin_excluida",
        "escala_estadisticas_m",
        "umbral_probabilidad_nube"
    ]

    sufijos = [
        "mean",
        "stdDev",
        "min",
        "max",
        "median",
        "p25",
        "p75",
        "count"
    ]

    columnas_estadisticas = [
        f"{indice}_{sufijo}"
        for indice in INDICES
        for sufijo in sufijos
    ]

    df_indices = df_indices.reindex(
        columns=columnas_identificacion + columnas_estadisticas
    )

    for columna in columnas_estadisticas:
        df_indices[columna] = pd.to_numeric(
            df_indices[columna],
            errors="raise"
        )

    display(
        df_indices[
            ["date", "image_id"]
            + [f"{indice}_mean" for indice in INDICES]
            + [f"{indice}_count" for indice in INDICES]
        ].head(10)
    )

    columnas_calidad = [
        "image_id",
        "pixeles_antes",
        "pixeles_despues",
        "porcentaje_conservado"
    ]

    df_indices = df_indices.merge(
        calidad_df[columnas_calidad],
        on="image_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    if not df_indices["_merge"].eq("both").all():
        raise ValueError(
            "Hay imágenes sin correspondencia en el resumen de calidad."
        )

    df_indices = df_indices.drop(
        columns="_merge"
    )

    print("Información de calidad incorporada.")

    import numpy as np

    assert len(df_indices) == numero_con_datos, (
        "El número de filas no coincide con las imágenes seleccionadas."
    )

    assert df_indices["image_id"].is_unique, (
        "Hay identificadores de imagen repetidos."
    )

    assert df_indices["parcela_id"].eq(
        parcela_actual["parcela_id"]
    ).all(), "Hay filas de otra parcela."

    assert df_indices["date"].ge(
        pd.Timestamp(inicio)
    ).all(), "Hay fechas anteriores al inicio."

    assert df_indices["date"].lt(
        pd.Timestamp(fin)
    ).all(), "Hay fechas fuera del límite final."

    assert df_indices["pixeles_despues"].gt(0).all(), (
        "Se ha incluido una imagen sin píxeles B8 válidos."
    )

    valores_estadisticos = df_indices[
        columnas_estadisticas
    ].to_numpy(dtype=float)

    assert not np.isinf(valores_estadisticos).any(), (
        "Hay valores infinitos en las estadísticas."
    )

    print("Comprobaciones finales superadas.")

    resumen_indices = pd.DataFrame([
        {
            "indice": indice,
            "imagenes_con_media": int(
                df_indices[f"{indice}_mean"].notna().sum()
            ),
            "imagenes_sin_media": int(
                df_indices[f"{indice}_mean"].isna().sum()
            ),
            "minimo_de_medias": df_indices[f"{indice}_mean"].min(),
            "maximo_de_medias": df_indices[f"{indice}_mean"].max()
        }
        for indice in INDICES
    ])

    display(resumen_indices)

    nombre_csv = (
        f"{parcela_actual['parcela_id']}_sentinel2_indices_"
        f"{inicio.isoformat()}_{fin.isoformat()}.csv"
    )

    archivo_csv = extraccion_path / nombre_csv

    df_indices.to_csv(
        archivo_csv,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print("CSV guardado:", archivo_csv.name)
    print("Filas:", len(df_indices))
    print("Fechas distintas:", df_indices["date"].nunique())
    print("Periodo provisional:", PERIODO_PROVISIONAL)

    def extraer_cobertura(elemento):
        image = ee.Image(elemento)

        proyeccion = (
            image.select("B8").projection()
        )

        area_pixel = ee.Image.pixelArea()

        parametros_reduccion = {
            "reducer": ee.Reducer.sum(),
            "geometry": polygon,
            "crs": proyeccion,
            "scale": parametros["escala_m"],
            "maxPixels": 1_000_000
        }

        area_referencia = (
            area_pixel
            .reduceRegion(**parametros_reduccion)
            .get("area")
        )

        propiedades = ee.Dictionary({
            "image_id": image.get("system:index"),
            "area_referencia_m2": area_referencia
        })

        for indice in INDICES:
            mascara_indice = (
                image.select(indice).mask()
            )

            area_valida = (
                area_pixel
                .updateMask(mascara_indice)
                .reduceRegion(**parametros_reduccion)
                .get("area")
            )

            propiedades = propiedades.set(
                f"{indice}_area_valida_m2",
                area_valida
            )

        return ee.Feature(
            None,
            propiedades
        )

    cobertura_fc = ee.FeatureCollection(
        coleccion_indices
        .toList(coleccion_indices.size())
        .map(extraer_cobertura)
    )

    datos_cobertura = cobertura_fc.getInfo()

    cobertura_df = pd.DataFrame([
        feature["properties"]
        for feature in datos_cobertura["features"]
    ])

    columnas_area = [
        "area_referencia_m2"
    ] + [
        f"{indice}_area_valida_m2"
        for indice in INDICES
    ]

    for columna in columnas_area:
        cobertura_df[columna] = pd.to_numeric(
            cobertura_df[columna],
            errors="raise"
        )

    if cobertura_df["area_referencia_m2"].isna().any():
        raise ValueError(
            "No se ha podido calcular alguna superficie de referencia."
        )

    if not cobertura_df["area_referencia_m2"].gt(0).all():
        raise ValueError(
            "La superficie de referencia debe ser positiva."
        )

    for indice in INDICES:
        columna_area = f"{indice}_area_valida_m2"
        columna_pct = f"{indice}_cobertura_pct"

        cobertura_df[columna_area] = (
            cobertura_df[columna_area].fillna(0)
        )

        cobertura_df[columna_pct] = (
            100
            * cobertura_df[columna_area]
            / cobertura_df["area_referencia_m2"]
        )

    columnas_cobertura = [
        f"{indice}_cobertura_pct"
        for indice in INDICES
    ]

    valores_cobertura = cobertura_df[
        columnas_cobertura
    ]

    if (
        (valores_cobertura < -0.01)
        | (valores_cobertura > 100.01)
    ).any().any():
        raise ValueError(
            "Hay porcentajes de cobertura fuera del intervalo esperado."
        )

    cobertura_df["cobertura_minima_indices_pct"] = (
        cobertura_df[columnas_cobertura].min(axis=1)
    )

    display(
        cobertura_df[
            ["image_id"] + columnas_cobertura
        ].head(10)
    )

    df_indices_calidad = df_indices.merge(
        cobertura_df,
        on="image_id",
        how="left",
        validate="one_to_one",
        indicator=True
    )

    if not df_indices_calidad["_merge"].eq("both").all():
        raise ValueError(
            "Hay imágenes sin correspondencia en la revisión de cobertura."
        )

    df_indices_calidad = (
        df_indices_calidad
        .drop(columns="_merge")
        .sort_values(["date", "image_id"])
        .reset_index(drop=True)
    )

    print("Observaciones revisadas:", len(df_indices_calidad))

    UMBRAL_COBERTURA_MINIMA_PCT = parametros["cobertura_minima_pct"]

    columnas_medias = [
        f"{indice}_mean"
        for indice in INDICES
    ]

    df_indices_calidad["umbral_cobertura_minima_pct"] = (
        UMBRAL_COBERTURA_MINIMA_PCT
    )

    df_indices_calidad["apta_cobertura"] = (
        df_indices_calidad["cobertura_minima_indices_pct"]
        .ge(UMBRAL_COBERTURA_MINIMA_PCT)
    )

    df_indices_calidad["medias_completas"] = (
        df_indices_calidad[columnas_medias]
        .notna()
        .all(axis=1)
    )

    df_indices_calidad["seleccion_tecnica_provisional"] = (
        df_indices_calidad["apta_cobertura"]
        & df_indices_calidad["medias_completas"]
    )

    display(
        df_indices_calidad[
            [
                "date",
                "cobertura_minima_indices_pct",
                "apta_cobertura",
                "seleccion_tecnica_provisional"
            ]
        ]
    )

    print(
        "Observaciones que superan el criterio:",
        int(
            df_indices_calidad[
                "seleccion_tecnica_provisional"
            ].sum()
        )
    )

    print(
        "Observaciones pendientes de revisión:",
        int(
            (~df_indices_calidad[
                "seleccion_tecnica_provisional"
            ]).sum()
        )
    )

    serie_seleccionada = (
        df_indices_calidad.loc[
            df_indices_calidad[
                "seleccion_tecnica_provisional"
            ]
        ]
        .copy()
        .sort_values("date")
    )

    fechas_repetidas = (
        serie_seleccionada["date"]
        .duplicated(keep=False)
    )

    if fechas_repetidas.any():
        detalle_repetidas = (
            serie_seleccionada.loc[
                fechas_repetidas,
                [
                    "date",
                    "image_id",
                    "cobertura_minima_indices_pct"
                ]
            ]
            .sort_values(["date", "image_id"])
        )

        display(detalle_repetidas)

        raise ValueError(
            f"La parcela {parcela_actual['nombre']} tiene "
            "varias imágenes seleccionadas en una misma fecha. "
            "Debe resolverse la selección por fecha antes "
            "de calcular el resumen semanal."
        )

    serie_seleccionada["dias_desde_anterior"] = (
        serie_seleccionada["date"]
        .diff()
        .dt.days
    )

    display(
        serie_seleccionada[
            [
                "date",
                "cobertura_minima_indices_pct",
                "dias_desde_anterior"
            ]
        ]
    )

    if len(serie_seleccionada) >= 2:
        print(
            "Mayor intervalo entre observaciones seleccionadas:",
            int(
                serie_seleccionada[
                    "dias_desde_anterior"
                ].max()
            ),
            "días"
        )
    else:
        print(
            "No hay suficientes observaciones seleccionadas "
            "para calcular intervalos."
        )

    archivo_calidad = extraccion_path / (
        f"{parcela_actual['parcela_id']}_sentinel2_indices_calidad_"
        f"{inicio.isoformat()}_{fin.isoformat()}.csv"
    )

    df_indices_calidad.to_csv(
        archivo_calidad,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print("CSV con revisión de calidad guardado:")
    print(archivo_calidad.name)

    semanal_df = crear_resumen_semanal(
        datos=df_indices_calidad,
        indices=INDICES,
        fecha_inicio=inicio,
        fecha_fin=fin
    )

    columnas_mostrar = [
        "semana_inicio",
        "n_imagenes_con_datos",
        "n_imagenes_seleccionadas",
        "cobertura_minima_utilizada_pct",
        "NDVI_media_semanal",
        "NDMI_media_semanal",
        "origen_dato"
    ]

    display(
        semanal_df[columnas_mostrar].round(3)
    )

    columnas_indices_semanales = [
        f"{indice}_media_semanal"
        for indice in INDICES
    ]

    sin_observaciones = (
        semanal_df["n_imagenes_seleccionadas"] == 0
    )

    assert semanal_df[
        "n_imagenes_seleccionadas"
    ].sum() == len(serie_seleccionada), (
        "El recuento semanal no coincide con las observaciones seleccionadas."
    )

    assert semanal_df.loc[
        sin_observaciones,
        columnas_indices_semanales
    ].isna().all().all(), (
        "Hay valores en semanas sin observaciones seleccionadas."
    )

    assert not semanal_df["interpolado"].any(), (
        "Esta versión no debe contener interpolación."
    )

    assert semanal_df["semana_inicio"].is_unique

    print("Comprobaciones semanales superadas.")
    print("Semanas del intervalo:", len(semanal_df))
    print(
        "Semanas con observaciones seleccionadas:",
        int((~sin_observaciones).sum())
    )
    print(
        "Semanas sin observaciones seleccionadas:",
        int(sin_observaciones.sum())
    )

    archivo_semanal = extraccion_path / (
        f"{parcela_actual['parcela_id']}_sentinel2_semanal_"
        f"{inicio.isoformat()}_{fin.isoformat()}.csv"
    )

    semanal_df.to_csv(
        archivo_semanal,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d"
    )

    print("Resumen semanal guardado:")
    print(archivo_semanal.name)

    if not archivo_semanal.exists():
        raise FileNotFoundError(
            "No se encuentra el CSV semanal."
        )

    if archivo_semanal.stat().st_size == 0:
        raise ValueError(
            "El CSV semanal está vacío."
        )

    semanal_guardado = pd.read_csv(
        archivo_semanal,
        encoding="utf-8-sig",
        parse_dates=[
            "semana_inicio",
            "semana_fin_excluida"
        ]
    )

    assert len(semanal_guardado) == len(semanal_df), (
        "El número de filas guardadas no coincide."
    )

    assert int(
        semanal_guardado["n_imagenes_seleccionadas"].sum()
    ) == len(serie_seleccionada), (
        "El CSV no conserva el recuento de observaciones."
    )

    semanas_vacias = (
        semanal_guardado["n_imagenes_seleccionadas"] == 0
    )

    assert semanal_guardado.loc[
        semanas_vacias,
        columnas_indices_semanales
    ].isna().all().all(), (
        "Las semanas sin observaciones deben conservar valores ausentes."
    )

    print("CSV semanal recuperado y comprobado.")
    print("Filas:", len(semanal_guardado))
    print("Semanas sin observaciones:", int(semanas_vacias.sum()))

    observaciones_seleccionadas = (
        df_indices_calidad.loc[
            df_indices_calidad[
                "seleccion_tecnica_provisional"
            ]
        ]
        .sort_values("date")
    )

    intervalos_dias = (
        observaciones_seleccionadas["date"]
        .diff()
        .dt.days
        .dropna()
    )

    mayor_intervalo = (
        int(intervalos_dias.max())
        if not intervalos_dias.empty
        else None
    )

    resumen_ejecucion = {
        "parcela_id": parcela_actual["parcela_id"],
        "nombre": parcela_actual["nombre"],
        "fecha_inicio": inicio.isoformat(),
        "fecha_fin_excluida": fin.isoformat(),
        "periodo_provisional": PERIODO_PROVISIONAL,
        "imagenes_originales": numero_originales,
        "imagenes_emparejadas": numero_emparejadas,
        "imagenes_con_pixeles_validos": int(
            (calidad_df["pixeles_despues"] > 0).sum()
        ),
        "imagenes_sin_pixeles_validos": int(
            (calidad_df["pixeles_despues"] == 0).sum()
        ),
        "observaciones_seleccionadas": len(
            observaciones_seleccionadas
        ),
        "observaciones_no_seleccionadas": int(
            (
                ~df_indices_calidad[
                    "seleccion_tecnica_provisional"
                ]
            ).sum()
        ),
        "mayor_intervalo_dias": mayor_intervalo,
        "semanas_totales": len(semanal_df),
        "semanas_con_observaciones": int(
            semanal_df[
                "tiene_observaciones_seleccionadas"
            ].sum()
        ),
        "semanas_sin_observaciones": int(
            (
                ~semanal_df[
                    "tiene_observaciones_seleccionadas"
                ]
            ).sum()
        )
    }

    resumen_ejecucion_df = pd.DataFrame(
        [resumen_ejecucion]
    )

    resumen_ejecucion_df.to_csv(
        extraccion_path / "resumen_ejecucion.csv",
        index=False,
        encoding="utf-8-sig"
    )

    display(resumen_ejecucion_df.T)

    resumen_ejecucion["umbral_nube"] = parametros["umbral_nube"]
    resumen_ejecucion["cobertura_minima_pct"] = parametros["cobertura_minima_pct"]
    resumen_ejecucion["escala_m"] = parametros["escala_m"]

    return {
        "resumen": resumen_ejecucion,
        "indices_calidad": df_indices_calidad,
        "semanal": semanal_df,
        "carpeta": extraccion_path
    }

## 12. Ejecución secuencial de todas las parcelas

Reinicia resultados_lote y procesa cada parcela. Ante el primer fallo se detiene y muestra la parcela afectada. Solo construye comparativa_df cuando devuelve resultados para todo el catálogo; no combina deliberadamente resultados nuevos con archivos antiguos de otras parcelas. La función escribe resultados individuales durante el proceso, por lo que no es una transacción de todo o nada.


In [ ]:
resultados_lote = {}
comparativa_df = None
for parcela in parcelas_lote:
    parcela_id = parcela["parcela_id"]
    try:
        resultados_lote[parcela_id] = procesar_parcela(
            parcela, FECHA_INICIO, FECHA_FIN, PARAMETROS, base_path
        )
    except Exception as error:
        raise RuntimeError(
            f"Lote detenido en {parcela_id}. No se genera una comparativa completa."
        ) from error

if set(resultados_lote) != set(catalogo["parcela_id"]):
    raise RuntimeError("No se han completado todas las parcelas.")
comparativa_df = pd.DataFrame([
    resultado["resumen"] for resultado in resultados_lote.values()
])
display(comparativa_df)


## 13. Guardado de la comparativa del lote

Verifica que exista una tabla con tantas filas como parcelas y guarda comparativa_parcelas.csv en la carpeta común del periodo. Ejecutar solo después de que termine correctamente el lote actual. Un archivo antiguo podría seguir en Drive si un lote posterior falla; la fecha de sus resultados debe revisarse.


In [ ]:
if comparativa_df is None or len(comparativa_df) != len(catalogo):
    raise RuntimeError("No existe una comparativa completa de esta ejecución.")
tipo_periodo = "provisional" if PARAMETROS["periodo_provisional"] else "confirmado"
comparativa_path = (
    base_path / "resultados" / "sentinel2_lotes" / tipo_periodo
    / "comparativas" / f"{inicio.isoformat()}_{fin.isoformat()}"
)
comparativa_path.mkdir(parents=True, exist_ok=True)
archivo_comparativa = comparativa_path / "comparativa_parcelas.csv"
comparativa_df.to_csv(archivo_comparativa, index=False, encoding="utf-8-sig")
print("Comparativa completa guardada:", archivo_comparativa)


## 14. Diagnóstico de teselas de Puri

Lee las tablas del lote en memoria, extrae el identificador de tesela del image_id y compara imágenes con fechas distintas. Este bloque es de diagnóstico: no cambia la selección ni vuelve a consultar Earth Engine. Varias teselas de una fecha no se deben eliminar sin revisar cobertura y criterio de agregación.


In [ ]:
puri_df = (
    resultados_lote["puri"]["indices_calidad"]
    .copy()
)

puri_df["date"] = pd.to_datetime(
    puri_df["date"],
    errors="raise"
)

puri_df["tesela_id"] = (
    puri_df["image_id"]
    .str.extract(r"_(T[0-9]{2}[A-Z]{3})$", expand=False)
)

puri_seleccionada = puri_df.loc[
    puri_df["seleccion_tecnica_provisional"].eq(True)
].copy()

print("PURI — OBSERVACIONES CON DATOS")
print("Imágenes:", len(puri_df))
print("Fechas distintas:", puri_df["date"].nunique())

print("\nPURI — OBSERVACIONES SELECCIONADAS")
print("Imágenes:", len(puri_seleccionada))
print(
    "Fechas distintas:",
    puri_seleccionada["date"].nunique()
)

print("\nTESELAS IDENTIFICADAS")
display(
    puri_df["tesela_id"]
    .value_counts(dropna=False)
    .rename_axis("tesela_id")
    .reset_index(name="numero_imagenes")
)

## 15. Revisión detallada de fechas de Puri

Cuenta imágenes disponibles, seleccionadas y teselas por fecha. Si varias imágenes superan la selección en una misma fecha, las muestra. No se presume que cada imagen represente una fecha independiente.


In [ ]:
resumen_fechas_puri = (
    puri_df
    .groupby("date")
    .agg(
        imagenes_con_datos=("image_id", "size"),
        imagenes_seleccionadas=(
            "seleccion_tecnica_provisional",
            "sum"
        ),
        teselas_distintas=("tesela_id", "nunique")
    )
    .reset_index()
    .sort_values("date")
)

display(resumen_fechas_puri)

fechas_repetidas_seleccionadas = (
    resumen_fechas_puri.loc[
        resumen_fechas_puri[
            "imagenes_seleccionadas"
        ].gt(1),
        "date"
    ]
)

print(
    "Fechas con más de una imagen seleccionada:",
    len(fechas_repetidas_seleccionadas)
)

if not fechas_repetidas_seleccionadas.empty:
    display(
        puri_seleccionada.loc[
            puri_seleccionada["date"].isin(
                fechas_repetidas_seleccionadas
            ),
            [
                "date",
                "image_id",
                "tesela_id",
                "cobertura_minima_indices_pct",
                "NDVI_mean",
                "NDMI_mean"
            ]
        ].sort_values(["date", "image_id"])
    )

## 16. Control de unicidad temporal en las cuatro parcelas

Comprueba en resultados_lote que ninguna fecha tenga varias imágenes seleccionadas para una misma parcela. Esta es una comprobación adicional del resultado del lote, además de la protección incorporada antes del resumen semanal. No exige que las imágenes descartadas sean únicas por fecha.


In [ ]:
revision_fechas = []

for parcela_id, resultado in resultados_lote.items():
    datos = resultado["indices_calidad"].copy()

    seleccionados = datos.loc[
        datos["seleccion_tecnica_provisional"].eq(True)
    ].copy()

    seleccionados["date"] = pd.to_datetime(
        seleccionados["date"],
        errors="raise"
    )

    imagenes_por_fecha = (
        seleccionados.groupby("date").size()
    )

    revision_fechas.append({
        "parcela_id": parcela_id,
        "imagenes_seleccionadas": len(seleccionados),
        "fechas_distintas": seleccionados["date"].nunique(),
        "fechas_con_varias_imagenes": int(
            imagenes_por_fecha.gt(1).sum()
        )
    })

revision_fechas_df = pd.DataFrame(
    revision_fechas
)

display(revision_fechas_df)

if revision_fechas_df[
    "fechas_con_varias_imagenes"
].gt(0).any():
    raise ValueError(
        "Hay parcelas con varias imágenes seleccionadas por fecha."
    )

print(
    "Comprobación superada: como máximo una imagen "
    "seleccionada por parcela y fecha."
)

## 17. Exportación de la revisión temporal

Guarda revision_fechas_seleccionadas.csv junto a la comparativa. Ejecutar después de construir comparativa_path y superar el control de fechas. No requiere repetir consultas remotas.


In [ ]:
revision_fechas_df.to_csv(
    comparativa_path / "revision_fechas_seleccionadas.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Revisión de fechas seleccionadas guardada.")

## Cierre y límites de interpretación
Conservar configuración, revisiones, índices, resumen semanal, comparativa y revisión de fechas. Los huecos no se rellenan en esta versión. La comparación de disponibilidad no indica qué parcela necesita más riego. Antes de conectar con el motor de decisión habrá que confirmar campaña, cultivo y las entradas agronómicas necesarias. Si se cambia el periodo o los umbrales, revisar nuevamente cobertura, fechas y continuidad temporal.
